In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('ggplot')



In [7]:
df_5to7 = pd.read_csv('data/accidents_2005_to_2007.csv', dtype={'LSOA_of_Accident_Location': str})
df_9to11 = pd.read_csv('data/accidents_2009_to_2011.csv', dtype={'LSOA_of_Accident_Location': str})
df_12to14 = pd.read_csv('data/accidents_2012_to_2014.csv', dtype={'LSOA_of_Accident_Location': str})

df_aadf = pd.read_csv('data/ukTrafficAADF.csv')


df_5to7.head(5)

,Accident_Index,Location_Easting_OSGR,Location_Northing_OSGR,Longitude,Latitude,Police_Force,Accident_Severity,Number_of_Vehicles,Number_of_Casualties,Date,...,Pedestrian_Crossing-Physical_Facilities,Light_Conditions,Weather_Conditions,Road_Surface_Conditions,Special_Conditions_at_Site,Carriageway_Hazards,Urban_or_Rural_Area,Did_Police_Officer_Attend_Scene_of_Accident,LSOA_of_Accident_Location,Year
0,200501BS00001,525680.0,178240.0,-0.191170,51.489096,1,2,1,1,04/01/2005,...,Zebra crossing,Daylight: Street light present,Raining without high winds,Wet/Damp,NaN,NaN,1,Yes,E01002849,2005
1,200501BS00002,524170.0,181650.0,-0.211708,51.520075,1,3,1,1,05/01/2005,...,Pedestrian phase at traffic signal junction,Darkness: Street lights present and lit,Fine without high winds,Dry,NaN,NaN,1,Yes,E01002909,2005
2,200501BS00003,524520.0,182240.0,-0.206458,51.525301,1,3,2,1,06/01/2005,...,No physical crossing within 50 meters,Darkness: Street lights present and lit,Fine without high winds,Dry,NaN,NaN,1,Yes,E01002857,2005
3,200501BS00004,526900.0,177530.0,-0.173862,51.482442,1,3,1,1,07/01/2005,...,No physical crossing within 50 meters,Daylight: Street light present,Fine without high winds,Dry,NaN,NaN,1,Yes,E01002840,2005
4,200501BS00005,528060.0,179040.0,-0.156618,51.495752,1,3,1,1,10/01/2005,...,No physical crossing within 50 meters,Darkness: Street lighting unknown,Fine without high winds,Wet/Damp,NaN,NaN,1,Yes,E01002863,2005


In [9]:
check = df_5to7.columns.equals(df_9to11.columns) and df_9to11.columns.equals(df_12to14.columns)

print(f"All files have the same features: {check}")

All files have the same features: True


In [37]:
df_total = pd.concat([df_5to7, df_9to11, df_12to14], ignore_index= True)

In [38]:
df_total.info()

<class 'pandas.DataFrame'>
RangeIndex: 1504150 entries, 0 to 1504149
Data columns (total 33 columns):
 #   Column                                       Non-Null Count    Dtype  
---  ------                                       --------------    -----  
 0   Accident_Index                               1504150 non-null  str    
 1   Location_Easting_OSGR                        1504049 non-null  float64
 2   Location_Northing_OSGR                       1504049 non-null  float64
 3   Longitude                                    1504049 non-null  float64
 4   Latitude                                     1504049 non-null  float64
 5   Police_Force                                 1504150 non-null  int64  
 6   Accident_Severity                            1504150 non-null  int64  
 7   Number_of_Vehicles                           1504150 non-null  int64  
 8   Number_of_Casualties                         1504150 non-null  int64  
 9   Date                                         1504150 non-

In [39]:
df_total.isnull().sum()

Accident_Index                                       0
Location_Easting_OSGR                              101
Location_Northing_OSGR                             101
Longitude                                          101
Latitude                                           101
Police_Force                                         0
Accident_Severity                                    0
Number_of_Vehicles                                   0
Number_of_Casualties                                 0
Date                                                 0
Day_of_Week                                          0
Time                                               117
Local_Authority_(District)                           0
Local_Authority_(Highway)                            0
1st_Road_Class                                       0
1st_Road_Number                                      0
Road_Type                                            0
Speed_limit                                          0
Junction_D

We have many columns with **missing values**. I will check on each of them to see if the `NaN` are representing `None` or if they are missing



In [40]:
df = df_total.drop(columns= ['Junction_Detail'])

In [41]:
feat_to_check = ['Junction_Control', 'Pedestrian_Crossing-Human_Control',
                  'Pedestrian_Crossing-Physical_Facilities', 
                  'Weather_Conditions', 'Road_Surface_Conditions',
                  'Special_Conditions_at_Site', 'Carriageway_Hazards',
                  'Did_Police_Officer_Attend_Scene_of_Accident',
                  'LSOA_of_Accident_Location'
                ]


for ele in feat_to_check:
    print('\n' + ('=' * 40))
    print(f"Feature: {ele}")
    print('-' * 40)
    print(df[ele].value_counts(dropna=False))



Feature: Junction_Control
----------------------------------------
Junction_Control
Giveway or uncontrolled     733940
NaN                         602835
Automatic traffic signal    155717
Stop Sign                     9179
Authorised person             2479
Name: count, dtype: int64

Feature: Pedestrian_Crossing-Human_Control
----------------------------------------
Pedestrian_Crossing-Human_Control
None within 50 metres                 1495269
Control by other authorised person       5220
Control by school crossing patrol        3644
NaN                                        17
Name: count, dtype: int64

Feature: Pedestrian_Crossing-Physical_Facilities
----------------------------------------
Pedestrian_Crossing-Physical_Facilities
No physical crossing within 50 meters          1252571
Pedestrian phase at traffic signal junction     100248
non-junction pedestrian crossing                 79231
Zebra crossing                                   40106
Central refuge                    

Checking on the *location* features, let see if all four location-related columns have missing values in the exact same rows

In [45]:
loc_cols = ['Location_Easting_OSGR', 'Location_Northing_OSGR', 'Longitude', 'Latitude']

null_mask = df[loc_cols].isnull()
is_same = df[loc_cols].isnull().all(axis=1).eq(df[loc_cols].isnull().any(axis=1)).all()

print(f"Are all location missing values in the same rows? {is_same}")

Are all location missing values in the same rows? True


Good based on the analytics and the findings, we can set up the imputation as:

- `Junction_Control`, `Special_Conditions_at_Site`, `Carriageway_Hazards`: NaN has high proportion. They represents *None* in this case
  
- `Pedestrian_Crossing-Human_Control`, `Pedestrian_Crossing-Physical_Facilities`, `Weather_Conditions`, `Road_Surface_Conditions`, `Time`, `Latitude` has minor proportion of NaN values **(< 1%)**, we will just drop them

- `Did_Police_Officer_Attend_Scene_of_Accident`, `LSOA_of_Accident_Location` has a bit more, so we will impute them as **unknown**

In [42]:
fill_none_feat = [
    'Junction_Control', 'Special_Conditions_at_Site',
    'Carriageway_Hazards',
]

drop_na_feat = [
    'Pedestrian_Crossing-Human_Control', 'Pedestrian_Crossing-Physical_Facilities',
    'Weather_Conditions', 'Road_Surface_Conditions', 'Time', 'Latitude'
]

fill_unknown_feat = [
    'Did_Police_Officer_Attend_Scene_of_Accident',
    'LSOA_of_Accident_Location'
]

In [43]:
for ele in fill_none_feat:
    df[ele] = df[ele].fillna('None')

df = df.dropna(subset=drop_na_feat)

for ele in fill_unknown_feat:
    df[ele] = df[ele].fillna('Unknown')

In [44]:
df.isnull().sum()

Accident_Index                                 0
Location_Easting_OSGR                          0
Location_Northing_OSGR                         0
Longitude                                      0
Latitude                                       0
Police_Force                                   0
Accident_Severity                              0
Number_of_Vehicles                             0
Number_of_Casualties                           0
Date                                           0
Day_of_Week                                    0
Time                                           0
Local_Authority_(District)                     0
Local_Authority_(Highway)                      0
1st_Road_Class                                 0
1st_Road_Number                                0
Road_Type                                      0
Speed_limit                                    0
Junction_Control                               0
2nd_Road_Class                                 0
2nd_Road_Number     